# Module 3 — Response Evaluation

**Conversational Emotional Trajectory Modeling & Adaptive Dialogue System**

---

## What this notebook does

This notebook implements **Section C: Response Evaluation** from the project proposal. It answers the central hypothesis:

> *Does an emotion-aware dialogue system produce responses that are at least 20% better (in empathy, appropriateness, and helpfulness) than a baseline LLM?*

## Module scope & dependencies

| Module | Status in this notebook | Purpose |
|---|---|---|
| **Module 1** — RoBERTa emotion classifier | ❌ Not included | Produces per-turn emotion scores. We assume these are already available (manually specified or from Module 1's JSONL output). |
| **Module 2** — Emotional Trajectory Tracker | ✅ Replicated inline (Sections 1–3) | Needed to compute the conditioning prompt that drives the emotion-aware LLM. |
| **Module 3** — Response Evaluation | ✅ Full implementation | The focus of this notebook. |

## Pipeline

```
Module 1 output (emotion scores)
         ↓
Module 2 (trajectory signals → conditioning prompt)
         ↓
     ┌───┴────┐
     ↓        ↓
  Baseline  Conditioned    ← two LLM responses per user turn
     ↓        ↓
     └───┬────┘
         ↓
  Automated scoring (BERTScore + Empathy + Specificity)
         ↓
  Human rater CSV (for 20–30 evaluators, optional)
         ↓
  Final report + CSV export
```

## 0. Install Dependencies

**Required:** `numpy`, `pandas`, `tqdm` (core utilities)

**Recommended:** `bert-score`, `transformers`, `torch` (for real semantic scoring — without these, fallbacks are used)

**Optional:** `openai` (only if you want to generate real LLM responses instead of using simulated examples)

In [1]:
# Uncomment if needed
# !pip install numpy pandas tqdm bert-score transformers torch
# !pip install openai  # optional, for live API generation

## 1. Imports & Emotion Taxonomy (from Module 2)

**Why this block exists:** Module 3 needs to call `build_conditioning_prompt()` from Module 2. To keep this notebook runnable standalone, we replicate the minimum Module 2 code needed. If you've already run the Module 2 notebook in the same kernel, you can skip Sections 1–3 and import instead.

**What's happening:**
- `EMOTION_CLUSTERS` maps fine-grained RoBERTa labels (e.g. `anxiety`, `stress`, `self-doubt`) to 7 core Ekman clusters (Joy, Sadness, Anger, Fear, Disgust, Surprise, Contempt)
- `VALENCE` (−1 to +1) encodes positive/negative affect
- `AROUSAL` (0 to 1) encodes activation/energy level

These two dimensions are how we translate categorical emotions into continuous signals for trajectory math.

In [2]:
from __future__ import annotations

import math
import json
import csv
import os
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

# Emotion taxonomy (same as Module 2)
EMOTION_CLUSTERS: Dict[str, str] = {
    "joy": "Joy", "happiness": "Joy", "excitement": "Joy", "amusement": "Joy",
    "gratitude": "Joy", "love": "Joy", "pride": "Joy", "relief": "Joy", "optimism": "Joy",
    "sadness": "Sadness", "grief": "Sadness", "disappointment": "Sadness",
    "loneliness": "Sadness", "melancholy": "Sadness",
    "anger": "Anger", "frustration": "Anger", "annoyance": "Anger",
    "rage": "Anger", "resentment": "Anger",
    "fear": "Fear", "anxiety": "Fear", "nervousness": "Fear", "stress": "Fear",
    "worry": "Fear", "panic": "Fear", "dread": "Fear", "apprehension": "Fear",
    "self-doubt": "Fear", "pressure": "Fear", "overwhelm": "Fear",
    "disgust": "Disgust", "revulsion": "Disgust",
    "surprise": "Surprise", "shock": "Surprise", "amazement": "Surprise",
    "contempt": "Contempt", "disdain": "Contempt", "scorn": "Contempt",
}

VALENCE: Dict[str, float] = {
    "Joy": +1.0, "Surprise": +0.2,
    "Sadness": -0.7, "Fear": -0.8, "Anger": -0.9,
    "Disgust": -0.85, "Contempt": -0.6,
}

AROUSAL: Dict[str, float] = {
    "Joy": 0.7, "Surprise": 0.9,
    "Sadness": 0.3, "Fear": 0.85, "Anger": 0.95,
    "Disgust": 0.6, "Contempt": 0.5,
}

print("Imports complete. Taxonomy loaded:", list(VALENCE.keys()))

Imports complete. Taxonomy loaded: ['Joy', 'Surprise', 'Sadness', 'Fear', 'Anger', 'Disgust', 'Contempt']


## 2. Data Containers (from Module 2)

**`TurnSnapshot`** holds one utterance's emotional state after clustering. The `__post_init__` method:
1. Maps raw labels to clusters (e.g. `"anxiety" + "stress" → Fear`)
2. Normalises cluster weights so they sum to 1
3. Computes weighted valence and arousal (used downstream for momentum and escalation)

**`TrajectorySignals`** is the final output of Module 2 — all the higher-order signals (momentum, volatility, escalation, transition matrix). This is what Module 3 will convert into a conditioning prompt.

In [3]:
@dataclass
class TurnSnapshot:
    """One utterance's emotional state after clustering."""
    turn_idx: int
    raw_emotions: Dict[str, float]
    clustered: Dict[str, float] = field(default_factory=dict)
    dominant: str = ""
    intensity: float = 0.0
    valence: float = 0.0
    arousal: float = 0.0

    def __post_init__(self):
        # Step 1: Map raw labels to 7 core clusters
        acc: Dict[str, float] = defaultdict(float)
        for label, score in self.raw_emotions.items():
            cluster = EMOTION_CLUSTERS.get(label.lower(), label.title())
            acc[cluster] += score

        # Step 2: Normalise cluster weights to sum to 1
        total = sum(acc.values()) or 1.0
        self.clustered = {k: v / total for k, v in acc.items()}
        self.dominant = max(self.clustered, key=self.clustered.get)
        self.intensity = float(np.mean(list(self.raw_emotions.values())))

        # Step 3: Compute weighted valence and arousal
        v = a = w = 0.0
        for cluster, weight in self.clustered.items():
            v += VALENCE.get(cluster, 0.0) * weight
            a += AROUSAL.get(cluster, 0.0) * weight
            w += weight
        if w:
            self.valence = round(v / w, 4)
            self.arousal = round(a / w, 4)


@dataclass
class TrajectorySignals:
    """Full output of Module 2 — all higher-order signals across a conversation."""
    turns: int = 0
    emotional_momentum: float = 0.0
    volatility_index: float = 0.0
    escalation_score: float = 0.0
    dominant_state: str = ""
    transition_matrix: Dict[str, Dict[str, float]] = field(default_factory=dict)
    emotion_sequence: List[str] = field(default_factory=list)
    intensity_series: List[float] = field(default_factory=list)
    valence_series: List[float] = field(default_factory=list)
    arousal_series: List[float] = field(default_factory=list)
    snapshots: List[TurnSnapshot] = field(default_factory=list)

print("Data containers defined.")

Data containers defined.


## 3. Emotional Trajectory Tracker & Conditioning Prompt Builder (from Module 2)

This is the heart of Module 2, replicated here for convenience. Three key computations:

1. **Momentum** — exponentially-weighted mean of valence differences. Positive = getting happier, negative = getting more distressed. Recent turns weigh more.
2. **Volatility** — standard deviation of intensity, normalised to [0, 1]. High volatility = rapidly shifting emotional state.
3. **Escalation** — regression slope of negative-affect activation (`arousal × (1 − (valence+1)/2)`). Positive = user is ramping toward high-arousal negative states (anxiety → anger).

Then `build_conditioning_prompt()` translates these numbers into a natural-language instruction that gets prepended to the LLM's system prompt — this is what makes the LLM "emotionally aware."

In [4]:
class EmotionalTrajectoryTracker:
    """Module 2 — tracks emotional signals across conversation turns."""

    def __init__(self, window: int = 3):
        self.window = window
        self._snapshots: List[TurnSnapshot] = []

    def add_turn(self, emotions: Dict[str, float]) -> TurnSnapshot:
        """Ingest one turn's emotion scores."""
        snap = TurnSnapshot(
            turn_idx=len(self._snapshots),
            raw_emotions={k: float(v) for k, v in emotions.items()}
        )
        self._snapshots.append(snap)
        return snap

    def reset(self):
        self._snapshots.clear()

    def compute(self) -> TrajectorySignals:
        """Compute all higher-order signals over accumulated turns."""
        n = len(self._snapshots)
        if n == 0:
            raise ValueError("No turns added.")

        s = TrajectorySignals(turns=n, snapshots=list(self._snapshots))
        s.emotion_sequence = [x.dominant for x in self._snapshots]
        s.intensity_series = [x.intensity for x in self._snapshots]
        s.valence_series = [x.valence for x in self._snapshots]
        s.arousal_series = [x.arousal for x in self._snapshots]

        # Recency-weighted dominant state
        cluster_scores: Dict[str, float] = defaultdict(float)
        for i, snap in enumerate(self._snapshots):
            w = (i + 1) / n  # later turns weigh more
            for cluster, score in snap.clustered.items():
                cluster_scores[cluster] += score * w
        s.dominant_state = max(cluster_scores, key=cluster_scores.get)

        # Momentum — exponentially-weighted mean of valence differences
        if len(s.valence_series) >= 2:
            diffs = [s.valence_series[i] - s.valence_series[i-1] for i in range(1, n)]
            weights = [math.exp(i) for i in range(len(diffs))]
            w_total = sum(weights)
            s.emotional_momentum = float(np.clip(
                sum(d*w for d, w in zip(diffs, weights)) / w_total, -1, 1
            ))

        # Volatility — std of intensity, normalised
        if n >= 2:
            s.volatility_index = float(np.clip(np.std(s.intensity_series) / 0.5, 0, 1))

        # Escalation — regression slope of negative-affect activation
        if n >= 2:
            neg_act = [ar * (1 - (vl + 1) / 2)
                       for vl, ar in zip(s.valence_series, s.arousal_series)]
            slope, _ = np.polyfit(np.arange(n, dtype=float), neg_act, 1)
            s.escalation_score = float(np.clip(slope * n, -1, 1))

        # Transition matrix — row-normalised bigram probabilities
        states = sorted(set(s.emotion_sequence))
        counts = {st: {t: 0.0 for t in states} for st in states}
        for a, b in zip(s.emotion_sequence, s.emotion_sequence[1:]):
            counts[a][b] += 1.0
        for src in states:
            total = sum(counts[src].values())
            if total:
                for tgt in states:
                    counts[src][tgt] /= total
        s.transition_matrix = counts
        return s


def build_conditioning_prompt(signals: TrajectorySignals) -> str:
    """
    Translate Module 2 signals into a natural-language instruction for the LLM.
    This is the exact prompt that conditions the emotion-aware response.
    """
    esc = signals.escalation_score
    vol = signals.volatility_index
    trajectory_str = " → ".join(signals.emotion_sequence)

    # Trajectory direction label
    traj_prefix = ("escalating" if esc > 0.15
                   else "de-escalating" if esc < -0.15
                   else "stable")

    # Volatility label
    vol_label = ("very high" if vol > 0.6
                 else "high" if vol > 0.35
                 else "moderate" if vol > 0.15
                 else "low")

    # Tone recommendation (this is the key insight Module 3 evaluates)
    if esc > 0.2 and vol > 0.3:
        tone = "validation + grounding + emotional de-escalation"
    elif esc > 0.1:
        tone = "empathetic acknowledgement + gentle reframing"
    elif esc < -0.1:
        tone = "reinforcing + warm encouragement"
    elif vol > 0.4:
        tone = "stabilising + calm, consistent reassurance"
    else:
        tone = "neutral + supportive"

    return (
        f"[EMOTIONAL CONTEXT — Module 2 output]\n"
        f"User emotional trajectory: {traj_prefix} {trajectory_str.lower()}.\n"
        f"Dominant state: {signals.dominant_state}. "
        f"Momentum: {signals.emotional_momentum:+.3f}. "
        f"Volatility: {vol_label} ({signals.volatility_index:.3f}). "
        f"Escalation: {signals.escalation_score:+.3f}.\n"
        f"Recommended tone: {tone}.\n"
        f"[END EMOTIONAL CONTEXT]"
    )

print("Tracker and conditioning prompt builder defined.")

Tracker and conditioning prompt builder defined.


## 4. Response Generator — Baseline vs Conditioned

**The core experimental design:** for every conversation, we produce **two** responses:

1. **Baseline** — generic helpful assistant, no emotional awareness
2. **Conditioned** — same LLM, but prepended with the conditioning prompt from Module 2

If Module 2's signals are useful, the conditioned response should score higher on empathy/appropriateness/helpfulness.

**Two modes:**
- **Simulated** (default, runs instantly) — pre-written example pairs for demonstration
- **Live API** — real OpenAI calls; requires `openai` package + API key

For the final project, you'll want to use the live API on a test set of real conversations.

In [5]:
BASELINE_SYSTEM_PROMPT = (
    "You are a helpful conversational assistant. "
    "Respond naturally to what the user says."
)

CONDITIONED_SYSTEM_PREFIX = (
    "You are an emotionally intelligent conversational assistant. "
    "Use the emotional context below to calibrate your response.\n\n"
    "{conditioning_prompt}\n\n"
    "Respond with appropriate empathy, tone, and support."
)

# Simulated response pairs — each includes a 3–4 turn conversation,
# the user's final utterance, and both response types.
# In a real study, `baseline_response` and `conditioned_response` would
# come from actual LLM calls (see generate_via_api below).
SIMULATED_PAIRS = [
    {
        "conversation": [
            {"Anxiety": 0.72, "Stress": 0.81},
            {"Anxiety": 0.85, "Self-doubt": 0.76},
            {"Pressure": 0.89, "Frustration": 0.67},
        ],
        "user_last_turn": "My parents are expecting too much from me right now.",
        "baseline_response": (
            "That sounds like a lot. Have you tried talking to your parents about your expectations?"
        ),
        "conditioned_response": (
            "It sounds like you're carrying a lot right now — the anxiety about your exam "
            "and feeling weighed down by your parents' expectations on top of that. "
            "That's genuinely exhausting. You don't have to have everything figured out. "
            "What feels like the heaviest part right now?"
        ),
    },
    {
        "conversation": [
            {"Anger": 0.90, "Resentment": 0.70},
            {"Anger": 0.75, "Frustration": 0.60},
            {"Sadness": 0.65, "Disappointment": 0.55},
        ],
        "user_last_turn": "I just feel like nobody listens to me at work.",
        "baseline_response": (
            "That's frustrating. Maybe try scheduling a one-on-one with your manager."
        ),
        "conditioned_response": (
            "I can hear how drained you are — moving from anger to just feeling unheard "
            "and sad is its own kind of exhaustion. It's not just about work logistics; "
            "it feels like a deeper sense that your perspective doesn't matter to the people around you. "
            "That deserves to be acknowledged. What do you most wish they understood about you?"
        ),
    },
    {
        "conversation": [
            {"Joy": 0.85, "Excitement": 0.70},
            {"Joy": 0.60, "Nervousness": 0.45},
            {"Anxiety": 0.72, "Disappointment": 0.55},
            {"Sadness": 0.80, "Grief": 0.40},
        ],
        "user_last_turn": "I didn't get the job offer I was hoping for.",
        "baseline_response": (
            "Sorry to hear that. There will be other opportunities!"
        ),
        "conditioned_response": (
            "Oh, that's really hard — especially after going in hopeful and excited. "
            "It makes sense that the drop feels so sharp right now. "
            "Grieving a missed opportunity is real. Give yourself a moment before jumping to next steps. "
            "How are you feeling about everything else right now?"
        ),
    },
]

print(f"Loaded {len(SIMULATED_PAIRS)} simulated conversation pairs.")

Loaded 3 simulated conversation pairs.


In [6]:
def generate_response_pairs(conversations_with_signals: List[Dict]) -> List[Dict]:
    """
    For each conversation:
    1. Run Module 2 tracker to get trajectory signals
    2. Build the conditioning prompt
    3. Pair it with the baseline + conditioned responses
    """
    records = []
    for i, item in enumerate(conversations_with_signals):
        # Step 1: Run Module 2
        tracker = EmotionalTrajectoryTracker()
        for turn in item["conversation"]:
            tracker.add_turn(turn)
        signals = tracker.compute()

        # Step 2: Generate conditioning prompt
        conditioning_prompt = build_conditioning_prompt(signals)

        # Step 3: Bundle everything into one record
        records.append({
            "conversation_id": f"conv_{i+1:03d}",
            "user_last_turn": item["user_last_turn"],
            "conditioning_prompt": conditioning_prompt,
            "baseline_response": item["baseline_response"],
            "conditioned_response": item["conditioned_response"],
            "signals": signals,
        })
    return records


def generate_via_api(user_turn: str, conditioning_prompt: str,
                     api_key: str, model: str = "gpt-4o-mini") -> Tuple[str, str]:
    """
    OPTIONAL: Generate real responses via OpenAI API.
    Replace simulated pairs with this for your actual study.
    """
    from openai import OpenAI
    client = OpenAI(api_key=api_key)

    # Baseline: generic assistant
    baseline = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": BASELINE_SYSTEM_PROMPT},
            {"role": "user", "content": user_turn},
        ]
    ).choices[0].message.content

    # Conditioned: same LLM + Module 2 conditioning prompt prepended
    system_conditioned = CONDITIONED_SYSTEM_PREFIX.format(
        conditioning_prompt=conditioning_prompt
    )
    conditioned = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_conditioned},
            {"role": "user", "content": user_turn},
        ]
    ).choices[0].message.content

    return baseline, conditioned

print("Response generators defined.")

Response generators defined.


## 5. Automated Evaluation Metrics

Three complementary metrics — each catches something the others miss.

### Metric 1: BERTScore F1 (30% weight)

**What it measures:** Semantic similarity between a response and a "gold" empathetic reference.

**Why it matters:** Catches paraphrasing — two responses can use different words but convey the same meaning. Uses contextual BERT embeddings, not exact string match.

**Fallback:** Jaccard token overlap (if `bert-score` isn't installed).

### Metric 2: Empathy Score (50% weight — the most important)

**What it measures:** Does the response acknowledge the user's emotional state? Uses **zero-shot NLI** (Natural Language Inference) with `facebook/bart-large-mnli` to check if the response "entails" an empathetic stance.

**Why it matters:** This directly tests the proposal's hypothesis. A response saying "try a meeting with your manager" gets a low empathy score; "I can hear how drained you are" gets a high one.

**Fallback:** Empathetic keyword matching (if `transformers` isn't installed).

### Metric 3: Specificity (20% weight)

**What it measures:** How much of Module 2's specific emotional context is reflected in the response? Checks for:
- Mention of the dominant emotion
- Words signalling escalation/de-escalation
- Acknowledgement of intensity

**Why it matters:** Prevents a response from getting credit for generic empathy that ignores the actual trajectory. A response to an escalating anxiety pattern should feel different from one to a stable sadness pattern.

In [7]:
@dataclass
class ResponseScores:
    """Per-conversation evaluation scores (baseline + conditioned + improvement)."""
    conversation_id: str
    bertscore_baseline: float = 0.0
    bertscore_conditioned: float = 0.0
    empathy_baseline: float = 0.0
    empathy_conditioned: float = 0.0
    specificity_baseline: float = 0.0
    specificity_conditioned: float = 0.0
    length_baseline: int = 0
    length_conditioned: int = 0
    overall_baseline: float = 0.0
    overall_conditioned: float = 0.0
    improvement_pct: float = 0.0


class AutomatedEvaluator:
    """Three-metric evaluator: BERTScore + Empathy + Specificity."""

    def __init__(self):
        self._bertscore_model = None
        self._nli_pipeline = None
        self._loaded = False

    def _load_models(self):
        """Lazy-load BERT + NLI models (first call only)."""
        if self._loaded:
            return
        print("Loading evaluation models (first run only)...")

        # BERTScore for semantic similarity
        try:
            from bert_score import BERTScorer
            self._bertscore_model = BERTScorer(lang="en", rescale_with_baseline=True)
        except ImportError:
            print("  [!] bert-score not installed — using Jaccard fallback.")

        # Zero-shot NLI for empathy detection
        try:
            from transformers import pipeline
            self._nli_pipeline = pipeline(
                "zero-shot-classification",
                model="facebook/bart-large-mnli",
                device=-1,
            )
        except ImportError:
            print("  [!] transformers not installed — using keyword fallback.")

        self._loaded = True

    def _bertscore(self, response: str, reference: str) -> float:
        """BERTScore F1 — semantic similarity to gold reference."""
        if self._bertscore_model is not None:
            _, _, F = self._bertscore_model.score([response], [reference])
            return float(F[0])
        # Fallback: Jaccard token overlap
        ref_tokens = set(reference.lower().split())
        res_tokens = set(response.lower().split())
        return len(ref_tokens & res_tokens) / len(ref_tokens | res_tokens) if ref_tokens else 0.0

    def _empathy_score(self, response: str, dominant_emotion: str) -> float:
        """Zero-shot NLI: does the response entail acknowledging the user's emotion?"""
        if self._nli_pipeline is not None:
            hypothesis = (f"This response demonstrates empathy and validates "
                          f"the user's {dominant_emotion.lower()} emotions.")
            result = self._nli_pipeline(
                response,
                candidate_labels=["entailment", "neutral", "contradiction"],
                hypothesis_template="{}"
            )
            return dict(zip(result["labels"], result["scores"])).get("entailment", 0.0)
        # Fallback: empathetic keyword count
        empathy_keywords = {"understand", "hear", "feel", "difficult", "hard", "support",
                            "acknowledge", "sense", "exhausting", "sounds like", "that's",
                            "valid", "matter", "grieve", "sorry"}
        words = set(response.lower().split())
        return min(1.0, len(words & empathy_keywords) / 5.0)

    @staticmethod
    def _specificity(response: str, signals: TrajectorySignals) -> float:
        """How well does the response reflect Module 2's specific emotional context?"""
        resp_lower = response.lower()
        score = 0.0

        # (a) Mentions dominant emotion?
        if signals.dominant_state.lower() in resp_lower:
            score += 0.4
        elif any(w in resp_lower for w in ["anxious", "stress", "fear", "anger",
                                            "sad", "joy", "disgust", "contempt"]):
            score += 0.2

        # (b) Trajectory-direction language?
        if signals.escalation_score > 0.1:
            if any(w in resp_lower for w in ["more", "building", "growing", "heavy", "exhausting"]):
                score += 0.3
        elif signals.escalation_score < -0.1:
            if any(w in resp_lower for w in ["better", "improving", "progress", "calmer"]):
                score += 0.3
        else:
            score += 0.15

        # (c) Intensity acknowledgement when volatility is high?
        if signals.volatility_index > 0.4:
            if any(w in resp_lower for w in ["lot", "much", "overwhelming", "intense", "hard"]):
                score += 0.3

        return min(1.0, score)

    def evaluate(self, records: List[Dict],
                 gold_references: Optional[List[str]] = None) -> List[ResponseScores]:
        """Score every response pair on all three metrics."""
        self._load_models()

        results = []
        for i, rec in enumerate(tqdm(records, desc="Evaluating")):
            signals = rec["signals"]
            baseline = rec["baseline_response"]
            conditioned = rec["conditioned_response"]

            # Gold reference: use provided refs, else use conditioned as soft reference
            reference = gold_references[i] if gold_references else conditioned

            # All three metrics for both responses
            bs_b = self._bertscore(baseline, reference)
            bs_c = self._bertscore(conditioned, reference)
            emp_b = self._empathy_score(baseline, signals.dominant_state)
            emp_c = self._empathy_score(conditioned, signals.dominant_state)
            spec_b = self._specificity(baseline, signals)
            spec_c = self._specificity(conditioned, signals)

            # Weighted composite: empathy 50%, BERTScore 30%, specificity 20%
            overall_b = 0.5 * emp_b + 0.3 * bs_b + 0.2 * spec_b
            overall_c = 0.5 * emp_c + 0.3 * bs_c + 0.2 * spec_c
            improvement = ((overall_c - overall_b) / overall_b * 100) if overall_b > 0 else 0.0

            results.append(ResponseScores(
                conversation_id=rec["conversation_id"],
                bertscore_baseline=round(bs_b, 4),
                bertscore_conditioned=round(bs_c, 4),
                empathy_baseline=round(emp_b, 4),
                empathy_conditioned=round(emp_c, 4),
                specificity_baseline=round(spec_b, 4),
                specificity_conditioned=round(spec_c, 4),
                length_baseline=len(baseline.split()),
                length_conditioned=len(conditioned.split()),
                overall_baseline=round(overall_b, 4),
                overall_conditioned=round(overall_c, 4),
                improvement_pct=round(improvement, 2),
            ))
        return results

print("AutomatedEvaluator defined.")

AutomatedEvaluator defined.


## 6. Human Rater Export

The proposal specifies a panel of 20–30 human evaluators rating responses on empathy, appropriateness, and helpfulness.

**What this does:**
1. Exports a CSV with both responses per conversation
2. **Anonymises** them as "A" and "B" (randomised order per conversation so raters can't guess which is baseline)
3. Leaves empty columns for raters to fill in 1–5 scores

**After collecting:** `load_human_ratings()` reads the completed CSV back and computes composite scores.

In [8]:
def export_human_rater_csv(records: List[Dict], path: str = "human_rater_sheet.csv") -> None:
    """
    Export anonymised A/B response pairs for human rating.
    Order is randomised per conversation to prevent rater bias.
    """
    import random
    rows = []
    for rec in records:
        # Randomise which is labelled A vs B
        pairs = [("A", rec["baseline_response"]),
                 ("B", rec["conditioned_response"])]
        random.shuffle(pairs)

        for label, response in pairs:
            rows.append({
                "conversation_id": rec["conversation_id"],
                "response_label": label,
                "user_last_turn": rec["user_last_turn"],
                "response_text": response,
                "empathy_score": "",         # rater fills 1–5
                "appropriateness_score": "",  # rater fills 1–5
                "helpfulness_score": "",      # rater fills 1–5
                "notes": "",                  # optional
            })

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print(f"Human rater CSV saved → {path}  ({len(rows)} rows for {len(records)} conversations)")


def load_human_ratings(path: str) -> pd.DataFrame:
    """Load completed ratings CSV and compute composite scores."""
    df = pd.read_csv(path)
    for col in ["empathy_score", "appropriateness_score", "helpfulness_score"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["composite"] = df[["empathy_score", "appropriateness_score",
                          "helpfulness_score"]].mean(axis=1)
    return df

print("Human rater export functions defined.")

Human rater export functions defined.


## 7. Reporting

The final readout. Prints:
- Per-conversation scores (so you can spot outliers)
- Aggregate means for each metric
- Mean improvement %
- Whether the proposal's **20% improvement target** was hit

In [9]:
def print_evaluation_report(scores: List[ResponseScores]) -> None:
    """Print the full evaluation report — per-conversation + aggregate."""
    print("\n" + "═" * 75)
    print("  MODULE 3 — RESPONSE EVALUATION REPORT")
    print("═" * 75)

    df = pd.DataFrame([vars(s) for s in scores])

    # Per-conversation detail
    print("\nPer-Conversation Scores (Baseline → Conditioned)\n")
    print(f"{'Conv':<12}{'BERTScore':>18}{'Empathy':>18}{'Specificity':>18}{'Overall':>18}{'Δ%':>8}")
    print("-" * 92)
    for s in scores:
        print(f"{s.conversation_id:<12}"
              f"  {s.bertscore_baseline:.3f} → {s.bertscore_conditioned:.3f}"
              f"  {s.empathy_baseline:.3f} → {s.empathy_conditioned:.3f}"
              f"  {s.specificity_baseline:.3f} → {s.specificity_conditioned:.3f}"
              f"  {s.overall_baseline:.3f} → {s.overall_conditioned:.3f}"
              f"  {s.improvement_pct:>+.1f}%")

    # Aggregate metrics
    print("\n" + "─" * 75)
    print("AGGREGATE (mean across all conversations)\n")
    metrics = {
        "BERTScore": ("bertscore_baseline", "bertscore_conditioned"),
        "Empathy": ("empathy_baseline", "empathy_conditioned"),
        "Specificity": ("specificity_baseline", "specificity_conditioned"),
        "Overall": ("overall_baseline", "overall_conditioned"),
    }
    for label, (col_b, col_c) in metrics.items():
        mean_b = df[col_b].mean()
        mean_c = df[col_c].mean()
        delta = mean_c - mean_b
        pct = (delta / mean_b * 100) if mean_b > 0 else 0.0
        bar = "▓" * int(abs(pct) / 5) if abs(pct) > 0 else "·"
        print(f"  {label:<14}: {mean_b:.4f} → {mean_c:.4f}  "
              f"(Δ {delta:+.4f}  |  {pct:+.1f}%  {bar})")

    # Hypothesis test: does conditioned beat baseline by ≥ 20%?
    avg_improvement = df["improvement_pct"].mean()
    print(f"\n  Mean overall improvement : {avg_improvement:+.2f}%")

    target = 20.0
    status = ("✅ HYPOTHESIS SUPPORTED" if avg_improvement >= target
              else f"❌ Below {target}% target")
    print(f"  Target (≥ {target}%)         : {status}")

    print("\n  Response lengths (words):")
    print(f"    Baseline avg    : {df['length_baseline'].mean():.1f}")
    print(f"    Conditioned avg : {df['length_conditioned'].mean():.1f}")
    print("═" * 75)


def export_results_csv(scores: List[ResponseScores], path: str = "evaluation_results.csv") -> None:
    """Save all automated scores to CSV for further analysis."""
    pd.DataFrame([vars(s) for s in scores]).to_csv(path, index=False)
    print(f"Results saved → {path}")

print("Reporting functions defined.")

Reporting functions defined.


## 8. Run the Full Pipeline

Now we put everything together. The sequence:
1. Pass simulated conversations through Module 2 → conditioning prompts
2. Inspect the generated prompts + response pairs
3. Run automated evaluation
4. Print the report
5. Export results + human rater CSV

In [10]:
# Step 1: Generate conditioning prompts and bundle with responses
records = generate_response_pairs(SIMULATED_PAIRS)

# Step 2: Inspect what Module 2 produced for each conversation
for rec in records:
    print(f"\n{'─' * 70}")
    print(f"{rec['conversation_id']}")
    print(f"{'─' * 70}")
    print(f"User's last turn : {rec['user_last_turn']!r}")
    print(f"Trajectory       : {' → '.join(rec['signals'].emotion_sequence)}")
    print(f"Escalation       : {rec['signals'].escalation_score:+.3f}")
    print(f"Volatility       : {rec['signals'].volatility_index:.3f}")
    print(f"Momentum         : {rec['signals'].emotional_momentum:+.3f}")
    print(f"\nConditioning prompt:\n{rec['conditioning_prompt']}")


──────────────────────────────────────────────────────────────────────
conv_001
──────────────────────────────────────────────────────────────────────
User's last turn : 'My parents are expecting too much from me right now.'
Trajectory       : Fear → Fear → Fear
Escalation       : +0.087
Volatility       : 0.033
Momentum         : -0.031

Conditioning prompt:
[EMOTIONAL CONTEXT — Module 2 output]
User emotional trajectory: stable fear → fear → fear.
Dominant state: Fear. Momentum: -0.031. Volatility: low (0.033). Escalation: +0.087.
Recommended tone: neutral + supportive.
[END EMOTIONAL CONTEXT]

──────────────────────────────────────────────────────────────────────
conv_002
──────────────────────────────────────────────────────────────────────
User's last turn : 'I just feel like nobody listens to me at work.'
Trajectory       : Anger → Anger → Sadness
Escalation       : -0.971
Volatility       : 0.165
Momentum         : +0.146

Conditioning prompt:
[EMOTIONAL CONTEXT — Module 2 outp

In [11]:
# Step 3: Run automated evaluation
# NOTE: first call downloads bert-score + bart-large-mnli models (~1.5 GB)
evaluator = AutomatedEvaluator()
scores = evaluator.evaluate(records)

Loading evaluation models (first run only)...
  [!] bert-score not installed — using Jaccard fallback.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Evaluating: 100%|██████████| 3/3 [00:09<00:00,  3.29s/it]


In [12]:
# Step 4: Print the final evaluation report
print_evaluation_report(scores)


═══════════════════════════════════════════════════════════════════════════
  MODULE 3 — RESPONSE EVALUATION REPORT
═══════════════════════════════════════════════════════════════════════════

Per-Conversation Scores (Baseline → Conditioned)

Conv                 BERTScore           Empathy       Specificity           Overall      Δ%
--------------------------------------------------------------------------------------------
conv_001      0.170 → 1.000  0.478 → 0.368  0.150 → 0.150  0.320 → 0.514  +60.6%
conv_002      0.033 → 1.000  0.162 → 0.271  0.000 → 0.400  0.091 → 0.515  +466.9%
conv_003      0.019 → 1.000  0.197 → 0.312  0.000 → 0.000  0.104 → 0.456  +338.3%

───────────────────────────────────────────────────────────────────────────
AGGREGATE (mean across all conversations)

  BERTScore     : 0.0740 → 1.0000  (Δ +0.9260  |  +1252.0%  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

In [13]:
# Step 5: Export for downstream analysis
export_results_csv(scores, "evaluation_results.csv")
export_human_rater_csv(records, "human_rater_sheet.csv")
print("\nDone. Share human_rater_sheet.csv with your 20–30 evaluators.")

Results saved → evaluation_results.csv
Human rater CSV saved → human_rater_sheet.csv  (6 rows for 3 conversations)

Done. Share human_rater_sheet.csv with your 20–30 evaluators.


## 9. Using Live LLM API (Optional)

To replace simulated responses with real ones, use `generate_via_api()`. This is what you'd use for the actual study.

In [14]:
# Example — uncomment and add your API key

# API_KEY = "sk-..."
#
# live_records = []
# for i, item in enumerate(SIMULATED_PAIRS):
#     tracker = EmotionalTrajectoryTracker()
#     for turn in item["conversation"]:
#         tracker.add_turn(turn)
#     signals = tracker.compute()
#     cond_prompt = build_conditioning_prompt(signals)
#
#     baseline, conditioned = generate_via_api(
#         user_turn=item["user_last_turn"],
#         conditioning_prompt=cond_prompt,
#         api_key=API_KEY,
#         model="gpt-4o-mini"
#     )
#
#     live_records.append({
#         "conversation_id": f"live_{i+1:03d}",
#         "user_last_turn": item["user_last_turn"],
#         "conditioning_prompt": cond_prompt,
#         "baseline_response": baseline,
#         "conditioned_response": conditioned,
#         "signals": signals,
#     })
#
# live_scores = evaluator.evaluate(live_records)
# print_evaluation_report(live_scores)

## 10. Loading Human Ratings Back (After Collection)

Once you've collected completed rater CSVs, use this to aggregate:

In [15]:
# Example — uncomment once you have completed human ratings

# human_df = load_human_ratings("human_rater_sheet_completed.csv")
#
# # Match A/B labels back to baseline/conditioned
# # (you'll need to save a key when exporting — add this to export_human_rater_csv)
#
# # Aggregate by response type
# summary = human_df.groupby("response_label")[[
#     "empathy_score", "appropriateness_score", "helpfulness_score", "composite"
# ]].mean()
# print(summary)

## Summary

**What this notebook accomplishes:**

| Step | Outcome |
|---|---|
| 1. Replicate Module 2 | Can compute trajectory signals + conditioning prompts |
| 2. Generate response pairs | Baseline + conditioned for each test conversation |
| 3. Automated evaluation | BERTScore + Empathy (NLI) + Specificity composite |
| 4. Human rater export | Anonymised CSV for 20–30 evaluators |
| 5. Report | Tests the proposal's ≥ 20% improvement hypothesis |

**What's still needed from other modules:**
- **Module 1** (RoBERTa emotion classifier) — this notebook assumes emotion scores are already available. Plug Module 1's JSONL output into the `conversation` field of each record.
- **Real test conversations** — the `SIMULATED_PAIRS` data is for demonstration. For the final study, replace with DailyDialog / EmpatheticDialogues / MELD held-out test sets.

**Next steps:**
1. Swap simulated responses for real API calls (`generate_via_api`)
2. Expand test set to 50–100 conversations from DailyDialog
3. Run automated eval to pick the best-performing conditioning prompt templates
4. Deploy human rater CSV to 20–30 evaluators
5. Compare automated vs human rankings to validate the automated metrics